## Notebook 2 - Baseline Model Developement and $$$ Value Estimation

This notebook will contain the following:

* Implement persistence, climatology mean, and simple linear regression baselines
* Compute FSS for each baseline to set the improvement bar
* Energy yield estimation: integrate GHI forecasts to estimate kWh/m² — this is your "business value" section
* Cost of forecast errors: link RMSE in W/m² to curtailment costs for a hypothetical solar farm

In [18]:

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

# Paths
DATA_PATH = Path("../data/01_raw/merged_data.csv")

MODEL_DIR = Path("../data/06_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("../data/08_reporting/notebook2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# City configuration
CITIES = ["bambili", "bamenda", "bafoussam", "yaounde"]

CITY_COLORS = {
    "bambili": "royalblue",
    "bamenda": "orange",
    "bafoussam": "green",
    "yaounde": "purple",
}

# Dataset columns
TARGET_COL = "irradiance"

BASE_FEATURES = [
    "temperature",
    "humidity",
    "wind_speed",
]

# Chronological split dates
TRAIN_END = "2010-12-31"
VAL_END = "2018-12-31"

print(" Notebook 2 initialized successfully")
print(f" Dataset path : {DATA_PATH}")
print(f" Models path  : {MODEL_DIR.resolve()}")
print(f" Output path  : {OUTPUT_DIR.resolve()}")

 Notebook 2 initialized successfully
 Dataset path : ..\data\01_raw\merged_data.csv
 Models path  : C:\Users\BRAIN\Desktop\ML\solar-irradiance-forecasting\data\06_models
 Output path  : C:\Users\BRAIN\Desktop\ML\solar-irradiance-forecasting\data\08_reporting\notebook2


In [19]:
merged_df = pd.read_csv(DATA_PATH)

# Parse date column
merged_df["date"] = pd.to_datetime(merged_df["date"])

# Sort chronologically
merged_df = merged_df.sort_values("date")

# Set date as index
merged_df = merged_df.set_index("date")

print("DATASET OVERVIEW")
print("═" * 60)

print(f"Rows          : {len(merged_df):,}")
print(f"Columns       : {list(merged_df.columns)}")
print(f"Cities        : {merged_df['city'].unique()}")
print(f"Date range    : {merged_df.index.min().date()} → {merged_df.index.max().date()}")
print(f"Missing values: {merged_df.isna().sum().sum():,}")

merged_df.head()

DATASET OVERVIEW
════════════════════════════════════════════════════════════
Rows          : 108,856
Columns       : ['temperature', 'humidity', 'irradiance', 'potential', 'wind_speed', 'city', 'month']
Cities        : ['bambili' 'bafoussam' 'yaounde' 'bamenda']
Date range    : 1950-01-01 → 2024-07-04
Missing values: 0


,temperature,humidity,irradiance,potential,wind_speed,city,month
date,,,,,,,
1950-01-01,20.3000,73.5360,637.5000,4.3901,0.2375,bambili,1
1950-01-01,22.3840,77.5130,805.1700,4.1294,0.1245,bafoussam,1
1950-01-01,24.5450,89.6320,650.8300,3.3640,1.5157,yaounde,1
1950-01-01,19.5750,74.5850,854.3300,4.3880,0.1986,bamenda,1
1950-01-02,20.3520,73.2250,837.5000,4.2901,0.2675,bambili,1


In [20]:
# SPLIT DATA BY CITY

dfs = {}

for city in CITIES:

    city_df = (
        merged_df[merged_df["city"] == city]
        .copy()
        .sort_index()
    )

    dfs[city] = city_df

    print(
        f"{city:<12} → "
        f"{len(city_df):>7,} rows | "
        f"{city_df.index.min().date()} → "
        f"{city_df.index.max().date()}"
    )

bambili      →  27,214 rows | 1950-01-01 → 2024-07-04
bamenda      →  27,214 rows | 1950-01-01 → 2024-07-04
bafoussam    →  27,214 rows | 1950-01-01 → 2024-07-04
yaounde      →  27,214 rows | 1950-01-01 → 2024-07-04


In [21]:
# ================================================================
# TIME SERIES SPLIT
# ================================================================
# IMPORTANT:
# Time series data must NEVER use random splitting.
#
# Train : Past
# Val   : Future relative to train
# Test  : Completely unseen future
# ================================================================

splits = {}

for city in CITIES:

    city_df = dfs[city]

    train_df = city_df.loc[:TRAIN_END]

    val_df = (
        city_df.loc[TRAIN_END:VAL_END]
        .iloc[1:]
    )

    test_df = (
        city_df.loc[VAL_END:]
        .iloc[1:]
    )

    splits[city] = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    print(
        f"{city:<12} | "
        f"train={len(train_df):>6,} | "
        f"val={len(val_df):>5,} | "
        f"test={len(test_df):>5,}"
    )

print("\n Chronological split completed")

bambili      | train=22,280 | val=2,922 | test=2,012
bamenda      | train=22,280 | val=2,922 | test=2,012
bafoussam    | train=22,280 | val=2,922 | test=2,012
yaounde      | train=22,280 | val=2,922 | test=2,012

 Chronological split completed


In [22]:
# ================================================================
# EVALUATION METRICS
# ================================================================
# MAE  : Mean Absolute Error
# RMSE : Root Mean Squared Error
# MAPE : Mean Absolute Percentage Error
# R²   : Coefficient of determination
# FSS  : Forecast Skill Score
        # FSS > 0 : Better than reference
        # FSS < 0 : Worse than reference
        # FSS = 1 : perfect forecast
# ================================================================

def compute_metrics(
    y_true,
    y_pred,
    y_reference=None,
    label="test"
):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    # Avoid division by zero
    valid = y_true > 1

    if valid.any():
        mape = np.mean(
            np.abs(
                (y_true[valid] - y_pred[valid])
                / y_true[valid]
            )
        ) * 100
    else:
        mape = np.nan

    # Forecast Skill Score
    if y_reference is not None:

        rmse_ref = np.sqrt(
            mean_squared_error(
                y_true,
                y_reference
            )
        )

    else:

        climatology = np.full_like(
            y_true,
            y_true.mean()
        )

        rmse_ref = np.sqrt(
            mean_squared_error(
                y_true,
                climatology
            )
        )

    fss = 1 - (rmse / (rmse_ref + 1e-8))

    return {
        f"{label}_mae": round(mae, 4),
        f"{label}_rmse": round(rmse, 4),
        f"{label}_mape": round(mape, 4),
        f"{label}_r2": round(r2, 4),
        f"{label}_fss": round(fss, 4),
    }


def print_metrics(model_name, metrics):

    print("\n" + "─" * 55)
    print(f"MODEL : {model_name}")
    print("─" * 55)

    for key, value in metrics.items():

        if "mae" in key or "rmse" in key:
            unit = " W/m²"

        elif "mape" in key:
            unit = " %"

        else:
            unit = ""

        print(f"{key:<20}: {value:>10.4f}{unit}")


print(" Metric helper functions ready")

 Metric helper functions ready


In [23]:
# ================================================================
# BASELINE 1 — PERSISTENCE MODEL
# ================================================================
# Assumption:
# Tomorrow's irradiance ≈ Today's irradiance
#
# ŷ(t) = y(t-1)
# ================================================================

persistence_results = {}

for city in CITIES:

    test_df = splits[city]["test"].copy()

    full_series = dfs[city][TARGET_COL]

    # Yesterday's irradiance
    test_df["prediction"] = (
        full_series
        .shift(1)
        .reindex(test_df.index)
    )

    test_df = test_df.dropna()

    y_true = test_df[TARGET_COL]
    y_pred = test_df["prediction"]

    climatology_reference = np.full_like(
        y_true,
        splits[city]["train"][TARGET_COL].mean()
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        y_reference=climatology_reference
    )

    persistence_results[city] = {
        "metrics": metrics,
        "y_true": y_true,
        "y_pred": y_pred,
        "index": test_df.index,
    }

    print_metrics(
        f"Persistence — {city.capitalize()}",
        metrics
    )

print("\n Persistence baseline completed")


───────────────────────────────────────────────────────
MODEL : Persistence — Bambili
───────────────────────────────────────────────────────
test_mae            :    95.1668 W/m²
test_rmse           :   135.6019 W/m²
test_mape           :    19.0651 %
test_r2             :     0.4127
test_fss            :     0.2367

───────────────────────────────────────────────────────
MODEL : Persistence — Bamenda
───────────────────────────────────────────────────────
test_mae            :    95.0708 W/m²
test_rmse           :   134.8938 W/m²
test_mape           :    19.5246 %
test_r2             :     0.4466
test_fss            :     0.2598

───────────────────────────────────────────────────────
MODEL : Persistence — Bafoussam
───────────────────────────────────────────────────────
test_mae            :   101.4148 W/m²
test_rmse           :   142.6687 W/m²
test_mape           :    19.2992 %
test_r2             :     0.2503
test_fss            :     0.1379

─────────────────────────────────────

In [24]:
# ================================================================
# BASELINE 2 — MONTHLY CLIMATOLOGY MODEL
# ================================================================
# Predict average irradiance for each calendar month.
#
# Example:
# All January values averaged together.
# ================================================================

climatology_results = {}

for city in CITIES:

    train_df = splits[city]["train"].copy()
    test_df = splits[city]["test"].copy()

    # Month extraction
    train_df["month"] = train_df.index.month
    test_df["month"] = test_df.index.month

    # Monthly averages from TRAIN ONLY
    monthly_climatology = (
        train_df
        .groupby("month")[TARGET_COL]
        .mean()
    )

    # Apply to test data
    test_df["prediction"] = (
        test_df["month"]
        .map(monthly_climatology)
    )

    y_true = test_df[TARGET_COL]
    y_pred = test_df["prediction"]

    persistence_reference = (
        dfs[city][TARGET_COL]
        .shift(1)
        .reindex(test_df.index)
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        y_reference=persistence_reference
    )

    climatology_results[city] = {
        "metrics": metrics,
        "y_true": y_true,
        "y_pred": y_pred,
        "index": test_df.index,
    }

    print_metrics(
        f"Climatology — {city.capitalize()}",
        metrics
    )

print("\nClimatology baseline completed")


───────────────────────────────────────────────────────
MODEL : Climatology — Bambili
───────────────────────────────────────────────────────
test_mae            :    92.8708 W/m²
test_rmse           :   119.1446 W/m²
test_mape           :    18.2974 %
test_r2             :     0.5466
test_fss            :     0.1214

───────────────────────────────────────────────────────
MODEL : Climatology — Bamenda
───────────────────────────────────────────────────────
test_mae            :    94.0206 W/m²
test_rmse           :   119.9869 W/m²
test_mape           :    18.8137 %
test_r2             :     0.5622
test_fss            :     0.1105

───────────────────────────────────────────────────────
MODEL : Climatology — Bafoussam
───────────────────────────────────────────────────────
test_mae            :    93.8492 W/m²
test_rmse           :   120.1780 W/m²
test_mape           :    17.7735 %
test_r2             :     0.4680
test_fss            :     0.1576

─────────────────────────────────────

In [25]:
# ================================================================
# FEATURE ENGINEERING
# ================================================================
# Daily dataset → daily lag features
#
# lag1  = yesterday
# lag7  = same day last week
# lag30 = previous month
# ================================================================

def create_features(df, full_series):

    df = df.copy()

    # Lag features
    df["irradiance_lag1"] = (
        full_series.shift(1)
        .reindex(df.index)
    )

    df["irradiance_lag7"] = (
        full_series.shift(7)
        .reindex(df.index)
    )

    df["irradiance_lag30"] = (
        full_series.shift(30)
        .reindex(df.index)
    )

    # Cyclical encoding
    df["month_sin"] = np.sin(
        2 * np.pi * df.index.month / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df.index.month / 12
    )

    df["dayofyear_sin"] = np.sin(
        2 * np.pi * df.index.dayofyear / 365
    )

    df["dayofyear_cos"] = np.cos(
        2 * np.pi * df.index.dayofyear / 365
    )

    return df

print("Feature engineering function ready")

Feature engineering function ready


In [26]:
# ================================================================
# BASELINE 3 — RIDGE REGRESSION
# ================================================================

lr_results = {}

for city in CITIES:

    train_df = splits[city]["train"].copy()
    val_df = splits[city]["val"].copy()
    test_df = splits[city]["test"].copy()

    full_series = dfs[city][TARGET_COL]

    train_df = create_features(train_df, full_series)
    val_df = create_features(val_df, full_series)
    test_df = create_features(test_df, full_series)

    FEATURES = [
        "temperature",
        "humidity",
        "wind_speed",
        "irradiance_lag1",
        "irradiance_lag7",
        "irradiance_lag30",
        "month_sin",
        "month_cos",
        "dayofyear_sin",
        "dayofyear_cos",
    ]

    train_df = train_df.dropna()
    val_df = val_df.dropna()
    test_df = test_df.dropna()

    X_train = train_df[FEATURES]
    y_train = train_df[TARGET_COL]

    X_val = val_df[FEATURES]
    y_val = val_df[TARGET_COL]

    X_test = test_df[FEATURES]
    y_test = test_df[TARGET_COL]

    # Scaling
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    X_val_scaled = scaler.transform(X_val)

    X_test_scaled = scaler.transform(X_test)

    # Hyperparameter tuning
    best_alpha = None
    best_mae = np.inf

    for alpha in [0.01, 0.1, 1, 10, 100]:

        model = Ridge(alpha=alpha)

        model.fit(X_train_scaled, y_train)

        val_pred = model.predict(X_val_scaled)

        val_mae = mean_absolute_error(
            y_val,
            val_pred
        )

        if val_mae < best_mae:

            best_mae = val_mae
            best_alpha = alpha

    # Final model
    final_model = Ridge(alpha=best_alpha)

    final_model.fit(
        X_train_scaled,
        y_train
    )

    y_pred = final_model.predict(X_test_scaled)

    persistence_reference = (
        full_series.shift(1)
        .reindex(test_df.index)
    )

    metrics = compute_metrics(
        y_test,
        y_pred,
        y_reference=persistence_reference
    )

    lr_results[city] = {
        "metrics": metrics,
        "model": final_model,
        "scaler": scaler,
        "features": FEATURES,
        "coefficients": dict(
            zip(FEATURES, final_model.coef_)
        ),
        "y_true": y_test,
        "y_pred": y_pred,
        "index": test_df.index,
    }

    print(f"\n{city.upper()} — Best alpha = {best_alpha}")

    print_metrics(
        f"Ridge Regression — {city.capitalize()}",
        metrics
    )

print("\nRidge regression completed")


BAMBILI — Best alpha = 100

───────────────────────────────────────────────────────
MODEL : Ridge Regression — Bambili
───────────────────────────────────────────────────────
test_mae            :    68.1762 W/m²
test_rmse           :    89.0344 W/m²
test_mape           :    13.9536 %
test_r2             :     0.7468
test_fss            :     0.3434

BAMENDA — Best alpha = 100

───────────────────────────────────────────────────────
MODEL : Ridge Regression — Bamenda
───────────────────────────────────────────────────────
test_mae            :    68.8143 W/m²
test_rmse           :    89.2313 W/m²
test_mape           :    14.3285 %
test_r2             :     0.7579
test_fss            :     0.3385

BAFOUSSAM — Best alpha = 100

───────────────────────────────────────────────────────
MODEL : Ridge Regression — Bafoussam
───────────────────────────────────────────────────────
test_mae            :    73.9483 W/m²
test_rmse           :    96.1755 W/m²
test_mape           :    14.4191 %
tes

In [27]:
# ================================================================
# BASELINE COMPARISON TABLE
# ================================================================

print("FORECAST SKILL SCORE COMPARISON")
print("═" * 75)

print(
    f"{'City':<15}"
    f"{'Persistence':>15}"
    f"{'Climatology':>15}"
    f"{'Ridge':>15}"
)

print("═" * 75)

for city in CITIES:

    p_fss = persistence_results[city]["metrics"]["test_fss"]

    c_fss = climatology_results[city]["metrics"]["test_fss"]

    lr_fss = lr_results[city]["metrics"]["test_fss"]

    print(
        f"{city.capitalize():<15}"
        f"{p_fss:>15.4f}"
        f"{c_fss:>15.4f}"
        f"{lr_fss:>15.4f}"
    )

print("═" * 75)

FORECAST SKILL SCORE COMPARISON
═══════════════════════════════════════════════════════════════════════════
City               Persistence    Climatology          Ridge
═══════════════════════════════════════════════════════════════════════════
Bambili                 0.2367         0.1214         0.3434
Bamenda                 0.2598         0.1105         0.3385
Bafoussam               0.1379         0.1576         0.3259
Yaounde                -0.0933         0.2203         0.4068
═══════════════════════════════════════════════════════════════════════════


In [28]:
# ================================================================
# PREDICTIONS VS ACTUALS
# ================================================================

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[c.capitalize() for c in CITIES]
)

positions = [(1,1), (1,2), (2,1), (2,2)]

WINDOW_START = "2022-01-01"
WINDOW_END = "2022-12-31"

for (row, col), city in zip(positions, CITIES):

    actual_series = pd.Series(
        lr_results[city]["y_true"],
        index=lr_results[city]["index"]
    )

    pred_series = pd.Series(
        lr_results[city]["y_pred"],
        index=lr_results[city]["index"]
    )

    actual_window = actual_series.loc[
        WINDOW_START:WINDOW_END
    ]

    pred_window = pred_series.loc[
        WINDOW_START:WINDOW_END
    ]

    fig.add_trace(
        go.Scatter(
            x=actual_window.index,
            y=actual_window.values,
            mode="lines",
            name="Actual",
            line=dict(color="black"),
            showlegend=(row == 1 and col == 1),
        ),
        row=row,
        col=col
    )

    fig.add_trace(
        go.Scatter(
            x=pred_window.index,
            y=pred_window.values,
            mode="lines",
            name="Prediction",
            line=dict(color=CITY_COLORS[city]),
            showlegend=(row == 1 and col == 1),
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Actual vs Predicted Irradiance (2022)",
    template="plotly_white",
    height=700,
)

fig.show()

fig.write_html(
    OUTPUT_DIR / "actual_vs_prediction.html"
)

print(" Prediction plot saved")

 Prediction plot saved


In [29]:
# ================================================================
# RESIDUAL ANALYSIS
# ================================================================

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[c.capitalize() for c in CITIES]
)

for (row, col), city in zip(positions, CITIES):

    residuals = (
        lr_results[city]["y_true"]
        - lr_results[city]["y_pred"]
    )

    fig.add_trace(
        go.Histogram(
            x=residuals,
            nbinsx=40,
            marker_color=CITY_COLORS[city],
            opacity=0.75,
            showlegend=False,
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Residual Distribution — Ridge Regression",
    template="plotly_white",
    height=700,
)

fig.show()

fig.write_html(
    OUTPUT_DIR / "residual_distribution.html"
)

print(" Residual analysis saved")

 Residual analysis saved


In [30]:
# ================================================================
# FEATURE COEFFICIENTS
# ================================================================

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[c.capitalize() for c in CITIES]
)

for (row, col), city in zip(positions, CITIES):

    coef_dict = lr_results[city]["coefficients"]

    features = list(coef_dict.keys())
    values = list(coef_dict.values())

    fig.add_trace(
        go.Bar(
            x=values,
            y=features,
            orientation="h",
            marker_color=CITY_COLORS[city],
            showlegend=False,
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Ridge Regression Feature Coefficients",
    template="plotly_white",
    height=700,
)

fig.show()

fig.write_html(
    OUTPUT_DIR / "feature_coefficients.html"
)

print("✓ Feature coefficient plot saved")

✓ Feature coefficient plot saved


In [31]:
# ================================================================
# ENERGY YIELD ESTIMATION
# ================================================================
# Convert irradiance to energy:
#
# Energy = Irradiance × daylight_hours
# ================================================================

DAYLIGHT_HOURS = 12

energy_results = {}

for city in CITIES:

    city_df = dfs[city].copy()

    city_df["energy_kwh_m2_day"] = (
        city_df[TARGET_COL]
        * DAYLIGHT_HOURS
        / 1000
    )

    annual_energy = (
        city_df
        .groupby(city_df.index.year)["energy_kwh_m2_day"]
        .sum()
    )

    energy_results[city] = {
        "daily_energy": city_df["energy_kwh_m2_day"],
        "annual_energy": annual_energy,
    }

    print(f"\n{city.upper()}")

    print(
        f"Mean daily energy  : "
        f"{city_df['energy_kwh_m2_day'].mean():.3f} kWh/m²/day"
    )

    print(
        f"Mean annual energy : "
        f"{annual_energy.mean():.2f} kWh/m²/year"
    )


BAMBILI
Mean daily energy  : 7.726 kWh/m²/day
Mean annual energy : 2803.31 kWh/m²/year

BAMENDA
Mean daily energy  : 7.574 kWh/m²/day
Mean annual energy : 2748.10 kWh/m²/year

BAFOUSSAM
Mean daily energy  : 7.888 kWh/m²/day
Mean annual energy : 2862.25 kWh/m²/year

YAOUNDE
Mean daily energy  : 6.707 kWh/m²/day
Mean annual energy : 2433.67 kWh/m²/year


In [32]:
# ================================================================
# ANNUAL ENERGY YIELD TRENDS
# ================================================================

fig = go.Figure()

for city in CITIES:

    annual_energy = (
        energy_results[city]["annual_energy"]
    )

    fig.add_trace(
        go.Scatter(
            x=annual_energy.index,
            y=annual_energy.values,
            mode="lines",
            name=city.capitalize(),
            line=dict(color=CITY_COLORS[city]),
        )
    )

fig.update_layout(
    title="Annual Solar Energy Yield",
    xaxis_title="Year",
    yaxis_title="Energy (kWh/m²/year)",
    template="plotly_white",
    height=500,
)

fig.show()

fig.write_html(
    OUTPUT_DIR / "annual_energy_yield.html"
)

print("Annual energy trend saved")

Annual energy trend saved


In [33]:
# ================================================================
# SAVE RESULTS
# ================================================================

summary = {}

for city in CITIES:

    summary[city] = {
        "Persistence": persistence_results[city]["metrics"],
        "Climatology": climatology_results[city]["metrics"],
        "RidgeRegression": lr_results[city]["metrics"],
    }

summary_path = OUTPUT_DIR / "baseline_summary.json"

with open(summary_path, "w") as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print(f"Summary saved → {summary_path}")

Summary saved → ..\data\08_reporting\notebook2\baseline_summary.json


In [34]:
merged_df[["irradiance", "potential"]].corr()

,irradiance,potential
irradiance,1.0000,1.0000
potential,1.0000,1.0000
